In [5]:
import cv2
import numpy as np
import tensorflow as tf
import mediapipe as mp
from tensorflow.keras.applications.efficientnet import preprocess_input


In [ ]:
# Load EfficientNet model
model = tf.keras.models.load_model('efficientNet_best_perc7.h5')
class_labels = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
IMG_SIZE = 224

# Inisialisasi MediaPipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.7)
mp_drawing = mp.solutions.drawing_utils

# Fungsi preprocessing khusus EfficientNet
def preprocess_hand(img):
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = preprocess_input(img.astype('float32'))
    return np.expand_dims(img, axis=0)

# Mulai kamera
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w, _ = frame.shape

    results = hands.process(rgb)
    boxes = []

    if results.multi_hand_landmarks:
        for landmarks in results.multi_hand_landmarks:
            x = [lm.x for lm in landmarks.landmark]
            y = [lm.y for lm in landmarks.landmark]
            xmin = int(min(x) * w) - 20
            xmax = int(max(x) * w) + 20
            ymin = int(min(y) * h) - 20
            ymax = int(max(y) * h) + 20

            xmin = max(0, xmin)
            ymin = max(0, ymin)
            xmax = min(w, xmax)
            ymax = min(h, ymax)

            boxes.append((xmin, ymin, xmax, ymax))

    if len(boxes) == 2:
        # Gabungkan dua tangan
        x1a, y1a, x2a, y2a = boxes[0]
        x1b, y1b, x2b, y2b = boxes[1]

        x_comb = min(x1a, x1b)
        y_comb = min(y1a, y1b)
        x2_comb = max(x2a, x2b)
        y2_comb = max(y2a, y2b)

        combined_img = frame[y_comb:y2_comb, x_comb:x2_comb]
        if combined_img.size > 0:
            pre = preprocess_hand(combined_img)
            pred = model.predict(pre, verbose=0)
            letter = class_labels[np.argmax(pred)]
            conf = np.max(pred) * 100

            cv2.rectangle(frame, (x_comb, y_comb), (x2_comb, y2_comb), (255, 0, 0), 2)
            cv2.putText(frame, f'{letter} ({conf:.1f}%)', (x_comb, y_comb - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    elif len(boxes) == 1:
        # Deteksi satu tangan saja
        x1, y1, x2, y2 = boxes[0]
        hand_img = frame[y1:y2, x1:x2]
        if hand_img.size > 0:
            pre = preprocess_hand(hand_img)
            pred = model.predict(pre, verbose=0)
            letter = class_labels[np.argmax(pred)]
            conf = np.max(pred) * 100

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f'{letter} ({conf:.1f}%)', (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    cv2.imshow("BISINDO Recognition - EfficientNet with MediaPipe", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
